In [1]:
SAMPLE_NAME = "ATQ3"
%env SAMPLE_NAME=ATQ3

env: SAMPLE_NAME=ATQ3


In [2]:
import os
os.chdir("..")
!pwd

/storage/scratch1/8/sverma87/Generative_Genomic_Data


In [ ]:
# Step 1: QC filtering (process .vcf.gz directly, autosomes only)

!plink2 \
  --vcf data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final.vcf.gz \
  --allow-extra-chr \
  --chr 1-22 \
  --mind 0.02 \
  --geno 0.02 \
  --hwe 1e-6 \
  --maf 0.01 \
  --max-alleles 2 \
  --split-par b38 \
  --make-bed \
  --out data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc

# Step 2: LD pruning - generate prune-in/prune-out SNP lists

!plink2 \
  --bfile data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc \
  --allow-extra-chr \
  --indep-pairwise 50 5 0.1 \
  --out data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc_ldprune

# Step 3: Extract LD-pruned SNPs and export as VCF

!plink2 \
  --bfile data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc \
  --allow-extra-chr \
  --extract data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc_ldprune.prune.in \
  --recode vcf bgz \
  --out data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc_ldpruned

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/ATQ3/ATQ3_qc2_grch38_final_qc.log.
Options in effect:
  --allow-extra-chr
  --chr 1-22
  --geno 0.02
  --hwe 1e-6
  --maf 0.01
  --make-bed
  --max-alleles 2
  --mind 0.02
  --out data/ATQ3/ATQ3_qc2_grch38_final_qc
  --split-par b38
  --vcf data/ATQ3/ATQ3_qc2_grch38_final.vcf.gz

Start time: Wed Jun 17 23:54:25 2026
1547242 MiB RAM detected, ~1476596 available; reserving 773621 MiB for main
workspace.
Using up to 192 threads (change this with --threads).
--vcf: 154635 variants scanned.
--vcf: data/ATQ3/ATQ3_qc2_grch38_final_qc-temporary.pgen +
data/ATQ3/ATQ3_qc2_grch38_final_qc-temporary.pvar.zst +
data/ATQ3/ATQ3_qc2_grch38_final_qc-temporary.psam written.
232 samples (0 females, 0 males, 232 ambiguous; 232 founders) loaded from
data/ATQ3/ATQ3_qc2_grch38_final_qc-temporary.psam.
154635 variants loaded from
data/ATQ

In [ ]:
# Step 4: Convert LD-pruned VCF to unphased genotype tensor (CSV)
!python src/vcf_to_unphased_tensor.py \
  --input  data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc_ldpruned.vcf.gz \
  --input-type vcf \
  --output data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_qc_ldpruned_unphased_tensor.csv

## Using Base Files for ATQ3 
I'm thinking that the files are already qc'd from Landon. We should just LD prune

In [7]:
# step 1: Generate Prune in files
!plink2 \
  --vcf data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final.vcf.gz \
  --allow-extra-chr \
  --indep-pairwise 50 5 0.1 \
  --out data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_ldprune
  
  
# Step 2: Extract LD-pruned SNPs and export as VCF
!plink2 \
  --vcf data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final.vcf.gz \
  --allow-extra-chr \
  --extract data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_ldprune.prune.in \
  --recode vcf bgz \
  --out data/"$SAMPLE_NAME"/"$SAMPLE_NAME"_qc2_grch38_final_ldpruned


PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/ATQ3/ATQ3_qc2_grch38_final_ldprune.log.
Options in effect:
  --allow-extra-chr
  --indep-pairwise 50 5 0.1
  --out data/ATQ3/ATQ3_qc2_grch38_final_ldprune
  --vcf data/ATQ3/ATQ3_qc2_grch38_final.vcf.gz

Start time: Thu Jun 18 00:12:28 2026
1547242 MiB RAM detected, ~1479827 available; reserving 773621 MiB for main
workspace.
Using up to 192 threads (change this with --threads).
--vcf: 154635 variants scanned.
--vcf: data/ATQ3/ATQ3_qc2_grch38_final_ldprune-temporary.pgen +
data/ATQ3/ATQ3_qc2_grch38_final_ldprune-temporary.pvar.zst +
data/ATQ3/ATQ3_qc2_grch38_final_ldprune-temporary.psam written.
232 samples (0 females, 0 males, 232 ambiguous; 232 founders) loaded from
data/ATQ3/ATQ3_qc2_grch38_final_ldprune-temporary.psam.
154635 variants loaded from
data/ATQ3/ATQ3_qc2_grch38_final_ldprune-temporary.pvar.zst.
Note: 

In [ ]:
# G